# Skin Condition Detector — Ensemble Backbones (Kaggle)

Train **EfficientNet-B0** and **MobileNetV3-Large** for the multi-model ensemble.
Your existing ResNet18 (`skin_model.pth`) stays as model 1 of 3.

## Fastest path (~2–3 hours GPU for both models)

1. Open [Skin Disease Dataset](https://www.kaggle.com/datasets/pacificrm/skindiseasedataset)
2. Click **New Notebook** on **that dataset page** (critical — attaches the data)
3. Upload this notebook, or copy/paste the cells
4. **Settings** → **Accelerator** → **GPU T4 x2**
5. Set which models to train in cell 1 (`TRAIN_BACKBONES`)
6. **Run All** → download the `.pth` files from the **Output** tab
7. Put them in your local `models/` folder next to `skin_model.pth`
8. Locally: `python scripts/evaluate_ensemble.py --data-dir …` then `--tune` (see `MODELING.md`)

> **Got `Dataset not found`?** The dataset is not attached. Use step 2 — don't create a blank notebook first.

Set `QUICK_MODE = True` in cell 1 for a faster (~40 min) smoke test.

Training uses **phone-like augmentations** (blur, JPEG noise, stronger color/crops) so models handle real captures better.

In [ ]:
import os
import re
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# True = fewer epochs for a quick test. False = full training.
QUICK_MODE = False

# Which backbones to train this run. Comment one out to train only one.
TRAIN_BACKBONES = [
    'efficientnet_b0',
    'mobilenet_v3_large',
]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'QUICK_MODE: {QUICK_MODE}')
print(f'TRAIN_BACKBONES: {TRAIN_BACKBONES}')

CANONICAL_CODES = [
    'acne', 'actinic_keratosis', 'benign_tumors', 'bullous', 'candidiasis',
    'drug_eruption', 'eczema', 'infestations_bites', 'lichen', 'lupus',
    'moles', 'psoriasis', 'rosacea', 'seborrheic_keratoses', 'skin_cancer',
    'sun_damage', 'tinea', 'normal', 'vascular_tumors', 'vasculitis',
    'vitiligo', 'warts',
]

FOLDER_TO_CODE = {
    'acne': 'acne',
    'actinic_keratosis': 'actinic_keratosis',
    'benign_tumors': 'benign_tumors',
    'bullous': 'bullous',
    'candidiasis': 'candidiasis',
    'drug_eruption': 'drug_eruption',
    'eczema': 'eczema',
    'infestations_bites': 'infestations_bites',
    'infestations': 'infestations_bites',
    'lichen': 'lichen',
    'lupus': 'lupus',
    'moles': 'moles',
    'psoriasis': 'psoriasis',
    'rosacea': 'rosacea',
    'seborrheic_keratoses': 'seborrheic_keratoses',
    'seborrh_keratoses': 'seborrheic_keratoses',
    'skin_cancer': 'skin_cancer',
    'skincancer': 'skin_cancer',
    'sun_sunlight_damage': 'sun_damage',
    'sun_damage': 'sun_damage',
    'tinea': 'tinea',
    'unknown_normal': 'normal',
    'normal': 'normal',
    'vascular_tumors': 'vascular_tumors',
    'vasculitis': 'vasculitis',
    'vitiligo': 'vitiligo',
    'warts': 'warts',
}

OUTPUT_NAMES = {
    'efficientnet_b0': 'skin_model_efficientnet_b0.pth',
    'mobilenet_v3_large': 'skin_model_mobilenet_v3_large.pth',
}

INPUT_DIR = Path('/kaggle/input')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_FOLDERS = {'train', 'test', 'val', 'validation', 'training', 'testing'}

if INPUT_DIR.exists():
    print('Mounted inputs:', [p.name for p in INPUT_DIR.iterdir()])
    for mount in INPUT_DIR.iterdir():
        if mount.is_dir():
            children = [c.name for c in mount.iterdir()][:8]
            print(f'  {mount.name}/ -> {children}')
else:
    print('WARNING: /kaggle/input does not exist — are you on Kaggle?')


def normalize_folder(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')


def folder_to_code(name: str) -> str:
    return FOLDER_TO_CODE.get(normalize_folder(name), normalize_folder(name))


def code_from_image_path(image_path: Path):
    """Match class from nearest folder name in the path (e.g. .../train/Acne/img.jpg)."""
    skip = SPLIT_FOLDERS | {'input', 'kaggle', 'working', 'data'}
    for part in reversed(image_path.parts):
        if part.lower() in skip:
            continue
        if part.startswith('skin') and 'disease' in part.lower():
            continue
        code = folder_to_code(part)
        if code in CANONICAL_CODES:
            return code
    return None


def scan_pacificrm_images():
    """Find all PacificRM images anywhere under /kaggle/input."""
    rows = []
    skipped_ham = 0
    unknown_folders = set()

    if not INPUT_DIR.exists():
        return rows

    for image_path in INPUT_DIR.rglob('*'):
        if not image_path.is_file():
            continue
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue
        if 'ham10000' in str(image_path).lower():
            skipped_ham += 1
            continue

        code = code_from_image_path(image_path)
        if code:
            rows.append({
                'image_path': str(image_path),
                'code': code,
                'source': 'pacificrm',
            })
        else:
            unknown_folders.add(image_path.parent.name)

    print(f'PacificRM images found: {len(rows)}')
    if rows:
        print('Sample path:', rows[0]['image_path'])
    if unknown_folders and not rows:
        print('Folders with images but no class match:', sorted(unknown_folders)[:10])
    if skipped_ham:
        print(f'(Skipped {skipped_ham} HAM10000 images)')
    return rows

In [ ]:
rows = scan_pacificrm_images()

if not rows:
    hint = """
    No images found. Fix:
    1. Open https://www.kaggle.com/datasets/pacificrm/skindiseasedataset
    2. Click NEW NOTEBOOK on that page (attaches dataset automatically)
       OR in your notebook: right sidebar → Add Input → search "Skin Disease Dataset"
    3. Re-run from the top

    Expected layout (either works):
      .../train/Acne/*.jpg
      .../test/Eczema/*.jpg
      OR .../Acne/*.jpg
    """
    raise FileNotFoundError(hint)

df = pd.DataFrame(rows)
code_to_idx = {code: i for i, code in enumerate(CANONICAL_CODES)}
df['label'] = df['code'].map(code_to_idx)
df = df.dropna(subset=['label']).reset_index(drop=True)

print(f'Total images: {len(df)}')
present = sorted(df['code'].unique())
missing = [c for c in CANONICAL_CODES if c not in present]
print(f'Classes found: {len(present)} / {len(CANONICAL_CODES)}')
if missing:
    print('Missing from dataset (no training images):', missing)
print(df['code'].value_counts())

In [ ]:
class RandomGaussianBlur:
    def __init__(self, p=0.25, radius=(0.1, 1.2)):
        self.p = p
        self.radius = radius

    def __call__(self, image):
        if np.random.rand() > self.p:
            return image
        from PIL import ImageFilter
        low, high = self.radius
        return image.filter(ImageFilter.GaussianBlur(radius=float(np.random.uniform(low, high))))


class RandomJPEGCompression:
    def __init__(self, p=0.3, quality=(40, 85)):
        self.p = p
        self.quality = quality

    def __call__(self, image):
        if np.random.rand() > self.p:
            return image
        import io
        low, high = self.quality
        buf = io.BytesIO()
        image.save(buf, format='JPEG', quality=int(np.random.randint(low, high + 1)))
        buf.seek(0)
        return Image.open(buf).convert('RGB')


# Stronger phone-like augs — improves real-world accuracy more than mild crops alone
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.02),
    RandomGaussianBlur(p=0.25),
    RandomJPEGCompression(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.12)),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SkinDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        return self.transform(image), int(row['label'])

BATCH_SIZE = 32
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

# Class-balanced sampling helps rarer conditions
label_counts = train_df['label'].value_counts().to_dict()
sample_weights = [1.0 / label_counts[int(lbl)] for lbl in train_df['label']]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    SkinDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
)
val_loader = DataLoader(
    SkinDataset(val_df, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

In [ ]:
def build_model(backbone, num_classes=22):
    if backbone == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
        return model
    if backbone == 'mobilenet_v3_large':
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model
    raise ValueError(f'Unsupported backbone: {backbone}')


def set_head_trainable(model, backbone):
    for param in model.parameters():
        param.requires_grad = False
    if backbone == 'efficientnet_b0':
        for param in model.classifier.parameters():
            param.requires_grad = True
    else:
        for param in model.classifier[-1].parameters():
            param.requires_grad = True


def train_epochs(model, loader, criterion, optimizer, epochs, label):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'[{label}] Epoch {epoch+1}/{epochs} — loss: {running_loss/len(loader):.4f}')


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = (all_preds == all_labels).mean()
    return acc, all_labels, all_preds


def train_one_backbone(backbone):
    print('\n' + '=' * 60)
    print(f'Training {backbone}')
    print('=' * 60)

    epochs_stage1 = 4 if QUICK_MODE else 8
    epochs_stage2 = 6 if QUICK_MODE else 15
    print(f'Stage 1 (head): {epochs_stage1} epochs | Stage 2 (full): {epochs_stage2} epochs')

    model = build_model(backbone, len(CANONICAL_CODES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # Stage 1: train head only
    set_head_trainable(model, backbone)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
    )
    train_epochs(model, train_loader, criterion, optimizer, epochs_stage1, f'{backbone}/head')

    # Stage 2: fine-tune all layers, keep best checkpoint
    for param in model.parameters():
        param.requires_grad = True
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_state = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(epochs_stage2):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        acc, _, _ = evaluate(model, val_loader)
        print(
            f'[{backbone}/full] Epoch {epoch+1}/{epochs_stage2} — '
            f'loss: {running_loss/len(train_loader):.4f} | val_acc: {acc:.2%}'
        )
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    acc, y_true, y_pred = evaluate(model, val_loader)
    print(f'\n{backbone} best validation accuracy: {acc:.2%}')
    print(classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CANONICAL_CODES))),
        target_names=CANONICAL_CODES,
        zero_division=0,
    ))

    output_path = f'/kaggle/working/{OUTPUT_NAMES[backbone]}'
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'class_codes': CANONICAL_CODES,
        'backbone': backbone,
        'dataset': 'pacificrm/skindiseasedataset',
        'num_classes': len(CANONICAL_CODES),
        'val_accuracy': float(best_acc),
    }
    torch.save(checkpoint, output_path)
    print(f'Saved {backbone} → {output_path}')
    return best_acc


print('Ready to train:', TRAIN_BACKBONES)

In [ ]:
results = {}
for backbone in TRAIN_BACKBONES:
    if backbone not in OUTPUT_NAMES:
        raise ValueError(f'Unknown backbone: {backbone}. Choose from {list(OUTPUT_NAMES)}')
    results[backbone] = train_one_backbone(backbone)

print('\n' + '=' * 60)
print('DONE — download these from the Output tab:')
for backbone, acc in results.items():
    print(f'  {OUTPUT_NAMES[backbone]}  (val_acc={acc:.2%})')
print('\nPut both files in your local models/ folder next to skin_model.pth')
print('Then restart the backend — /health should show models_loaded: 3')